# Practical 6: Object Detection / Transfer Learning Using a Pre-trained CNN

**Problem statement stages**
- **(a)** Load a pre-trained CNN trained on a large dataset.
- **(b)** Freeze weights in the lower convolutional layers.
- **(c)** Add a custom classifier with trainable parameters.
- **(d)** Train the classifier on task-specific data.
- **(e)** Fine-tune hyperparameters and unfreeze more layers as needed.

This notebook preserves the provided implementation: ResNet50 pretrained on ImageNet, MNIST digit classification, an initial frozen-base training stage, and fine-tuning of the final convolutional block.

**Note:** Although the syllabus labels this practical as object detection, the supplied code implements image classification using transfer learning, not bounding-box object detection. It is aligned with the listed transfer-learning stages.

## Imports and reproducibility

In [1]:
# Import required libraries

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.datasets import mnist
from IPython.display import display

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

## (b) Load and preprocess the dataset

In [2]:
# Load MNIST and select small subsets to reduce training time

(x_train, y_train), (x_test, y_test) = mnist.load_data()

rng = np.random.default_rng(0)

train_indices = rng.choice(len(x_train), 6000, replace=False)
test_indices = rng.choice(len(x_test), 1500, replace=False)

def preprocess_images(images):
    # Resize grayscale images to 32x32
    images = tf.image.resize(
        images[..., None].astype("float32"),
        (32, 32)
    ).numpy()

    # Convert grayscale images to 3-channel RGB
    images = np.repeat(images, 3, axis=-1)

    # Apply ResNet50 ImageNet preprocessing
    return preprocess_input(images)

X_tr = preprocess_images(x_train[train_indices])
y_tr = y_train[train_indices]

X_te = preprocess_images(x_test[test_indices])
y_te = y_test[test_indices]

print("Training data shape:", X_tr.shape)
print("Training labels shape:", y_tr.shape)
print("Testing data shape:", X_te.shape)
print("Testing labels shape:", y_te.shape)

Training data shape: (6000, 32, 32, 3)
Training labels shape: (6000,)
Testing data shape: (1500, 32, 32, 3)
Testing labels shape: (1500,)


## (a) Load the pre-trained CNN

In [3]:
# Load ResNet50 pretrained on ImageNet, without its original classification head

base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(32, 32, 3)
)

print("Base model layers:", len(base.layers))
print("Base model parameters:", f"{base.count_params():,}")

Base model layers: 175
Base model parameters: 23,587,712


## (b) Freeze the pre-trained convolutional base

In [4]:
# Freeze the pre-trained base so its weights do not update during classifier training

base.trainable = False

print("Pre-trained base frozen:", not base.trainable)

Pre-trained base frozen: True


## (c) Add the custom classifier and compile the model

In [5]:
# Add a task-specific classification head for the 10 MNIST digit classes

inputs = keras.Input(shape=(32, 32, 3))

x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

trainable_params = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
print("Trainable parameters (classifier head):", f"{trainable_params:,}")

model.summary()

Trainable parameters (classifier head): 527,114


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 32, 32, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ resnet50 (Functional)                │ (None, 1, 1, 2048)          │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 2048)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         524,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 24,114,826 (91.99 MB)

 Trainable params: 527,114 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

## (d) Train the custom classifier

In [ ]:
# Train the classifier while the pre-trained base remains frozen

h1 = model.fit(
    X_tr,
    y_tr,
    validation_data=(X_te, y_te),
    epochs=8,
    batch_size=64,
    verbose=2
)

stage1_loss, stage1_acc = model.evaluate(X_te, y_te, verbose=0)

print(f"\nStage-1 evaluation loss: {stage1_loss:.4f}")
print(f"Stage-1 evaluation accuracy: {stage1_acc:.4f}")

Epoch 1/8
94/94 - 145s - 2s/step - accuracy: 0.7370 - loss: 0.8525 - val_accuracy: 0.8793 - val_loss: 0.3672
Epoch 2/8


## (e) Unfreeze selected layers and fine-tune

In [ ]:
# Unfreeze the final convolutional block (conv5), while keeping Batch Normalization frozen

base.trainable = True

for layer in base.layers:
    layer.trainable = (
        layer.name.startswith("conv5")
        and not isinstance(layer, layers.BatchNormalization)
    )

# Recompile with a smaller learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

trainable_params = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
print("Trainable parameters after fine-tuning setup:", f"{trainable_params:,}")

In [ ]:
# Fine-tune the selected pre-trained layers along with the classifier head

h2 = model.fit(
    X_tr,
    y_tr,
    validation_data=(X_te, y_te),
    epochs=6,
    batch_size=64,
    verbose=2
)

## Evaluate the final model

In [ ]:
# Evaluate the model after fine-tuning

final_loss, final_acc = model.evaluate(X_te, y_te, verbose=0)

print(f"Final evaluation loss: {final_loss:.4f}")
print(f"Final evaluation accuracy: {final_acc:.4f}")

## Plot training and validation loss and accuracy

In [ ]:
# Combine the histories from frozen-base training and fine-tuning

loss_all = h1.history["loss"] + h2.history["loss"]
val_loss_all = h1.history["val_loss"] + h2.history["val_loss"]
acc_all = h1.history["accuracy"] + h2.history["accuracy"]
val_acc_all = h1.history["val_accuracy"] + h2.history["val_accuracy"]

stage1_epochs = len(h1.history["loss"])
epoch_numbers = range(1, len(loss_all) + 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# Loss graph
ax[0].plot(epoch_numbers, loss_all, label="Training loss")
ax[0].plot(epoch_numbers, val_loss_all, label="Validation loss")
ax[0].axvline(stage1_epochs + 0.5, linestyle="--", label="Fine-tuning starts")
ax[0].set_title("Loss: Frozen Base and Fine-tuning")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()
ax[0].grid(alpha=0.3)

# Accuracy graph
ax[1].plot(epoch_numbers, acc_all, label="Training accuracy")
ax[1].plot(epoch_numbers, val_acc_all, label="Validation accuracy")
ax[1].axvline(stage1_epochs + 0.5, linestyle="--", label="Fine-tuning starts")
ax[1].set_title("Accuracy: Frozen Base and Fine-tuning")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].legend()
ax[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig("a6_curves.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Graph saved as: a6_curves.png")

## Result and conclusion

In [ ]:
print("Practical 6 completed successfully.")
print("Stage 1: Frozen-base classifier training - 8 epochs")
print("Stage 2: Fine-tuning selected conv5 layers - 6 epochs")
print("Total training epochs:", len(loss_all))
print(f"Final evaluation accuracy: {final_acc:.4f}")

The transfer-learning model uses ResNet50 pretrained on ImageNet. Its pretrained layers are initially frozen, and a custom classification head is trained on the MNIST digit dataset. Selected layers in the final convolutional block are then unfrozen and fine-tuned with a lower learning rate. Training and validation curves and final evaluation metrics are used to inspect performance.

**Important:** This notebook implements transfer learning for image classification. It does not perform bounding-box object detection. If your instructor specifically requires detecting and localizing objects in images, a detector that predicts bounding boxes and class labels would be needed.